# 02 — Extract MIDRC ZIPs and inspect DICOMs

After notebook 01 succeeds, MIDRC delivers each series as a single ZIP
under `data/raw/<class>/<case_id>/<study_uid>/<series_uid>.zip`.

This notebook:
1. Extracts every ZIP into `data/extracted/`, preserving the class folder.
2. Reads DICOM headers (no pixel data) from the extracted tree.
3. Saves per-file metadata (PatientID-bearing, **gitignored**).
4. Groups by `SeriesInstanceUID` -> `outputs/sample_series_summary.csv`.
5. Prints counts so we can sanity-check that labels match the metadata
   *before* we scale up.

The actual logic lives in `scripts/extract_midrc_zips.py` and
`scripts/inspect_dicoms.py`; this notebook just imports/calls them so you can
poke the resulting DataFrames interactively.

In [2]:
import sys
from pathlib import Path

# Works whether the notebook is launched from the repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
RAW_DIR       = REPO_ROOT / "data" / "raw"
EXTRACTED_DIR = REPO_ROOT / "data" / "extracted"
OUT_DIR       = REPO_ROOT / "outputs"
PER_FILE_CSV   = OUT_DIR / "dicom_file_metadata.csv"
PER_SERIES_CSV = OUT_DIR / "sample_series_summary.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "scripts"))
import extract_midrc_zips as extractor
import inspect_dicoms as inspector

print("raw       :", RAW_DIR)
print("extracted :", EXTRACTED_DIR)
print("outputs   :", OUT_DIR)

raw       : /Users/lucaspu/HelperV1/data/raw
extracted : /Users/lucaspu/HelperV1/data/extracted
outputs   : /Users/lucaspu/HelperV1/outputs


## 1. Extract all ZIPs under `data/raw/` into `data/extracted/`

Already-extracted directories are skipped. Pass `--force` from the CLI (or
delete `data/extracted/` by hand) to force a re-extract.

In [3]:
extractor.main([
    "--input",  str(RAW_DIR),
    "--output", str(EXTRACTED_DIR),
])

2026-05-30 21:31:43,633 INFO extract_midrc_zips: Found 10 zip file(s) under /Users/lucaspu/HelperV1/data/raw
2026-05-30 21:31:43,649 INFO extract_midrc_zips: extracted 3 files -> /Users/lucaspu/HelperV1/data/extracted/chest_xray/10003752-1azc4d4F9UOj1vIbA3G0BQ/2.16.840.1.114274.1818.556972516376746133513059802808506623660/2.16.840.1.114274.1818.553519050903910273312094497357122423454
2026-05-30 21:31:43,672 INFO extract_midrc_zips: extracted 3 files -> /Users/lucaspu/HelperV1/data/extracted/chest_xray/10003752-w3uhjBebLUeBwtJdDSIA/2.16.840.1.114274.1818.54862041639390405042181213605039563172/2.16.840.1.114274.1818.54492409217557237789954425251006210218
2026-05-30 21:31:43,690 INFO extract_midrc_zips: extracted 1 files -> /Users/lucaspu/HelperV1/data/extracted/chest_xray/232451-001906/1.2.826.0.1.3680043.10.474.232451.68031/1.2.826.0.1.3680043.10.474.232451.68034
2026-05-30 21:31:43,702 INFO extract_midrc_zips: extracted 1 files -> /Users/lucaspu/HelperV1/data/extracted/chest_xray/41963


  zips found        : 10
  zips extracted    : 10 (others were skipped or already done)
  files on disk     : 252 (under /Users/lucaspu/HelperV1/data/extracted)
  first 10 extracted file paths:
    /Users/lucaspu/HelperV1/data/extracted/chest_xray/419639-004486/1.2.826.0.1.3680043.10.474.419639.270396332866013633460836136843/1.2.826.0.1.3680043.10.474.419639.567892396369263363143494437150/1.2.826.0.1.3680043.10.474.419639.567892396369263363143494437150/1.2.826.0.1.3680043.10.474.419639.880965775622024355330776381498.dcm
    /Users/lucaspu/HelperV1/data/extracted/chest_xray/514382-011600/1.2.826.0.1.3680043.10.474.514382.1826468/1.2.826.0.1.3680043.10.474.514382.1826469/1.2.826.0.1.3680043.10.474.514382.1826469/1.2.826.0.1.3680043.10.474.514382.1826467.dcm
    /Users/lucaspu/HelperV1/data/extracted/chest_xray/10003752-w3uhjBebLUeBwtJdDSIA/2.16.840.1.114274.1818.54862041639390405042181213605039563172/2.16.840.1.114274.1818.54492409217557237789954425251006210218/2.16.840.1.114274.1818.54

0

## 2. Inspect extracted DICOM headers

This will:
- recursively read every file under `data/extracted/`,
- skip non-DICOM / unreadable files with a warning,
- write `outputs/dicom_file_metadata.csv` (per-file, gitignored),
- write `outputs/sample_series_summary.csv` (per-series, safe to commit),
- print a summary.

In [4]:
inspector.main([
    "--input",  str(EXTRACTED_DIR),
    "--output", str(PER_SERIES_CSV),
])

2026-05-30 21:33:23,985 INFO inspect_dicoms: Scanning /Users/lucaspu/HelperV1/data/extracted ...
2026-05-30 21:33:23,993 INFO inspect_dicoms: Found 252 candidate files.
2026-05-30 21:33:24,127 INFO inspect_dicoms: Wrote per-file metadata -> /Users/lucaspu/HelperV1/outputs/dicom_file_metadata.csv
2026-05-30 21:33:24,137 INFO inspect_dicoms: Wrote per-series summary (10 series) -> /Users/lucaspu/HelperV1/outputs/sample_series_summary.csv



  readable DICOM files : 252
  unreadable files     : 0
  unique studies       : 10
  unique series        : 10

  counts by source_class:
    head_ct              243
    chest_xray           9

  counts by Modality:
    CT                   243
    CR                   7
    DX                   2

  counts by StudyDescription:
    BRAIN W/O CONTRAST (CT)-CS               243
    XR PORT CHEST 1V                         7
    XR CHEST 1 VIEW AP                       1
    XR CHEST 1 VW, FRONTAL                   1

  first rows of series summary:
source_class                                                 StudyInstanceUID                                                SeriesInstanceUID Modality BodyPartExamined           StudyDescription          SeriesDescription ViewPosition  number_of_files                                                                                                                                                                                                

0

## 3. Interactive look at the resulting DataFrames

Reload the CSVs we just wrote so we can slice them with pandas.

In [5]:
import pandas as pd

per_file_df = pd.read_csv(PER_FILE_CSV) if PER_FILE_CSV.exists() else pd.DataFrame()
series_df   = pd.read_csv(PER_SERIES_CSV) if PER_SERIES_CSV.exists() else pd.DataFrame()

print("per-file rows :", len(per_file_df))
print("series rows   :", len(series_df))
series_df.head(20)

per-file rows : 252
series rows   : 10


,source_class,StudyInstanceUID,SeriesInstanceUID,Modality,BodyPartExamined,StudyDescription,SeriesDescription,ViewPosition,number_of_files,example_path
0,chest_xray,1.2.826.0.1.3680043.10.474.232451.68031,1.2.826.0.1.3680043.10.474.232451.68034,CR,CHEST,XR PORT CHEST 1V,CXR AP GRID,AP,1,data/extracted/chest_xray/232451-001906/1.2.82...
1,chest_xray,1.2.826.0.1.3680043.10.474.419639.270396332866...,1.2.826.0.1.3680043.10.474.419639.567892396369...,DX,CHEST,XR CHEST 1 VIEW AP,CHEST AP,AP,1,data/extracted/chest_xray/419639-004486/1.2.82...
2,chest_xray,1.2.826.0.1.3680043.10.474.514382.1826468,1.2.826.0.1.3680043.10.474.514382.1826469,DX,PORT CHEST,"XR CHEST 1 VW, FRONTAL",AP,AP,1,data/extracted/chest_xray/514382-011600/1.2.82...
3,chest_xray,2.16.840.1.114274.1818.54862041639390405042181...,2.16.840.1.114274.1818.54492409217557237789954...,CR,CHEST,XR PORT CHEST 1V,ClearRead Bone Suppression,NaN,3,data/extracted/chest_xray/10003752-w3uhjBebLUe...
4,chest_xray,2.16.840.1.114274.1818.55697251637674613351305...,2.16.840.1.114274.1818.55351905090391027331209...,CR,CHEST,XR PORT CHEST 1V,CXR AP GRID,AP,3,data/extracted/chest_xray/10003752-1azc4d4F9UO...
5,head_ct,2.16.840.1.114274.1818.51427149410218196123597...,2.16.840.1.114362.1.12177026.25874454649.63540...,CT,NaN,BRAIN W/O CONTRAST (CT)-CS,BrainSegmentation,NaN,56,data/extracted/head_ct/10008204-z8khJe2hAkUFo6...
6,head_ct,2.16.840.1.114274.1818.47875515012030584817638...,2.16.840.1.114362.1.12177026.25874454649.63599...,CT,NaN,BRAIN W/O CONTRAST (CT)-CS,BrainSegmentation,NaN,32,data/extracted/head_ct/10008204-Rl4CqeLvpkB3r1...
7,head_ct,2.16.840.1.114274.1818.53399054345234663611660...,2.16.840.1.114362.1.12177026.25874454649.63630...,CT,NaN,BRAIN W/O CONTRAST (CT)-CS,BrainSegmentation,NaN,49,data/extracted/head_ct/10008204-FWKHNt4OoEI9ql...
8,head_ct,2.16.840.1.114274.1818.47534219802929417114523...,2.16.840.1.114362.1.12177026.25874454649.63677...,CT,NaN,BRAIN W/O CONTRAST (CT)-CS,BrainSegmentation,NaN,52,data/extracted/head_ct/10008204-OkR5yBUuQESa84...
9,head_ct,2.16.840.1.114274.1818.50293343243290883761489...,2.16.840.1.114362.1.12177026.25874454649.63773...,CT,NaN,BRAIN W/O CONTRAST (CT)-CS,BrainSegmentation,NaN,54,data/extracted/head_ct/10008204-rLWd9rqKkyQWjQ...


### Sanity-check: do the metadata labels match the intended class?

We pulled `chest_xray` using `modality in {CR, DX}` and `head_ct` using
`modality == CT`. If the per-series `Modality` column disagrees with the
folder name (`source_class`), that's a label-quality red flag worth
investigating before scaling up.

In [6]:
if not series_df.empty:
    cross = (
        series_df.groupby(["source_class", "Modality"], dropna=False)
                 .size()
                 .rename("n_series")
                 .reset_index()
    )
    print(cross.to_string(index=False))
else:
    print("No series found yet. Run cells above.")

source_class Modality  n_series
  chest_xray       CR         3
  chest_xray       DX         2
     head_ct       CT         5


In [7]:
if not series_df.empty:
    desc = (
        series_df.groupby(["source_class", "StudyDescription"], dropna=False)
                 .size()
                 .rename("n_series")
                 .reset_index()
                 .sort_values(["source_class", "n_series"], ascending=[True, False])
    )
    print(desc.to_string(index=False))

source_class           StudyDescription  n_series
  chest_xray           XR PORT CHEST 1V         3
  chest_xray         XR CHEST 1 VIEW AP         1
  chest_xray     XR CHEST 1 VW, FRONTAL         1
     head_ct BRAIN W/O CONTRAST (CT)-CS         5


### Done.

If the modality cross-tab looks right (CR/DX under `chest_xray`, CT under
`head_ct`) and `outputs/sample_series_summary.csv` has one row per imaging
series, this phase is complete. Do **not** download more data yet — next we
decide whether to scale up to a slightly larger sample or to iterate on the
filters.